# Part 16: Firehose and Repository Sync

The firehose is where ATProto becomes a continuously synchronized protocol
participant — events are ordered, sequenced, and replayable from cursors.
Repository sync uses MST diffs to bring consumers up to date.

**What you'll learn:**
- Event sequencing with monotonically increasing sequence numbers
- Subscriber management (subscribe, unsubscribe, deliver)
- Cursor-based replay for late subscribers
- MST snapshot diffing (add, update, delete operations)
- Applying diffs to synchronize repositories

**Prerequisites:** Parts 6-11 (ATProto primitives), Part 14 (record writes)

**Estimated Time:** 25-30 minutes

## Step 1: Firehose Event

Every event on the firehose has a sequence number, a type, and a DID.
In real implementations, `FirehoseCommitEvent` carries CBOR-encoded commit data.

In [ ]:
@interface FirehoseEvent : NSObject
@property (nonatomic, assign) int seq;
@property (nonatomic, strong) NSString *type;
@property (nonatomic, strong) NSString *did;
@property (nonatomic, strong) NSDictionary *data;
- (NSString *)description;
@end

@implementation FirehoseEvent
- (NSString *)description {
    return [NSString stringWithFormat:@"#%d [%@] %@", _seq, _type, _did];
}
@end

## Step 2: Event Sequencer

The sequencer assigns monotonically increasing sequence numbers.
`SubscribeReposHandler.nextSequenceNumber` does the same in the real codebase.

In [ ]:
@interface EventSequencer : NSObject
@property (nonatomic, assign) int currentSeq;
- (int)nextSequenceNumber;
- (FirehoseEvent *)createEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data;
@end

@implementation EventSequencer
- (instancetype)init {
    self = [super init];
    if (self) { _currentSeq = 0; }
    return self;
}
- (int)nextSequenceNumber {
    _currentSeq = _currentSeq + 1;
    return _currentSeq;
}
- (FirehoseEvent *)createEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data {
    FirehoseEvent *event = [[FirehoseEvent alloc] init];
    event.seq = [self nextSequenceNumber];
    event.type = type;
    event.did = did;
    event.data = data;
    return event;
}
@end

EventSequencer *seq = [[EventSequencer alloc] init];
NSLog(@"Seq: %d", [seq nextSequenceNumber]);
NSLog(@"Seq: %d", [seq nextSequenceNumber]);
NSLog(@"Seq: %d", [seq nextSequenceNumber]);

## Step 3: Firehose Subscriber

Each subscriber tracks its own cursor (the last seq it received).
This mirrors `WebSocketConnection` state in the real codebase.

In [ ]:
@interface FirehoseSubscriber : NSObject
@property (nonatomic, strong) NSString *subscriberId;
@property (nonatomic, assign) int cursor;
@property (nonatomic, strong) NSMutableArray *eventBuffer;
- (void)receiveEvent:(FirehoseEvent *)event;
- (NSMutableArray *)eventsSinceCursor:(int)sinceCursor;
@end

@implementation FirehoseSubscriber
- (instancetype)initWithId:(NSString *)sid {
    self = [super init];
    if (self) {
        _subscriberId = sid;
        _cursor = 0;
        _eventBuffer = [NSMutableArray array];
    }
    return self;
}
- (void)receiveEvent:(FirehoseEvent *)event {
    [_eventBuffer addObject:event];
    _cursor = event.seq;
}
- (NSMutableArray *)eventsSinceCursor:(int)sinceCursor {
    NSMutableArray *results = [NSMutableArray array];
    for (int i = 0; i < [_eventBuffer count]; i++) {
        FirehoseEvent *e = [_eventBuffer objectAtIndex:i];
        if (e.seq > sinceCursor) {
            [results addObject:e];
        }
    }
    return results;
}
@end

## Step 4: Firehose

The firehose manages subscribers, broadcasts events, and supports cursor-based replay.
This mirrors `SubscribeReposHandler` which maintains `attachedConnections` and
broadcasts `FirehoseCommitEvent` objects.

In [ ]:
@interface Firehose : NSObject
@property (nonatomic, strong) NSMutableArray *subscribers;
@property (nonatomic, strong) NSMutableArray *eventLog;
@property (nonatomic, strong) EventSequencer *sequencer;
- (void)subscribe:(FirehoseSubscriber *)subscriber;
- (void)unsubscribe:(NSString *)subscriberId;
- (FirehoseEvent *)broadcastEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data;
- (void)replayFromCursor:(int)cursor toSubscriber:(FirehoseSubscriber *)subscriber;
@end

@implementation Firehose
- (instancetype)init {
    self = [super init];
    if (self) {
        _subscribers = [NSMutableArray array];
        _eventLog = [NSMutableArray array];
        _sequencer = [[EventSequencer alloc] init];
    }
    return self;
}
- (void)subscribe:(FirehoseSubscriber *)subscriber {
    [_subscribers addObject:subscriber];
    NSLog(@"Subscribed: %@", subscriber.subscriberId);
}
- (void)unsubscribe:(NSString *)subscriberId {
    for (int i = 0; i < [_subscribers count]; i++) {
        FirehoseSubscriber *s = [_subscribers objectAtIndex:i];
        if ([[s subscriberId] isEqualToString:subscriberId]) {
            [_subscribers removeObjectAtIndex:i];
            NSLog(@"Unsubscribed: %@", subscriberId);
            return;
        }
    }
}
- (FirehoseEvent *)broadcastEventWithType:(NSString *)type did:(NSString *)did data:(NSDictionary *)data {
    FirehoseEvent *event = [_sequencer createEventWithType:type did:did data:data];
    [_eventLog addObject:event];
    // Deliver to all subscribers
    for (int i = 0; i < [_subscribers count]; i++) {
        FirehoseSubscriber *s = [_subscribers objectAtIndex:i];
        [s receiveEvent:event];
    }
    NSLog(@"Broadcast: %@", [event description]);
    return event;
}
- (void)replayFromCursor:(int)cursor toSubscriber:(FirehoseSubscriber *)subscriber {
    int count = 0;
    for (int i = 0; i < [_eventLog count]; i++) {
        FirehoseEvent *e = [_eventLog objectAtIndex:i];
        if (e.seq > cursor) {
            [subscriber receiveEvent:e];
            count = count + 1;
        }
    }
    NSLog(@"Replayed %d events from cursor %d to %@", count, cursor, subscriber.subscriberId);
}
@end

## Step 5: Full Firehose Demo

Subscribe two listeners, broadcast events, verify both receive them.
Unsubscribe one, broadcast more, verify only one receives.
Replay from cursor for a late subscriber.

In [ ]:
Firehose *fh = [[Firehose alloc] init];

// Subscribe two listeners
FirehoseSubscriber *sub1 = [[FirehoseSubscriber alloc] initWithId:@"relay-1"];
FirehoseSubscriber *sub2 = [[FirehoseSubscriber alloc] initWithId:@"appview-1"];
[fh subscribe:sub1];
[fh subscribe:sub2];

// Broadcast events
[fh broadcastEventWithType:@"commit" did:@"did:plc:abc" data:@{@"rev": @"1"}];
[fh broadcastEventWithType:@"commit" did:@"did:plc:def" data:@{@"rev": @"2"}];

NSLog(@"Sub1 cursor: %d, events: %d", sub1.cursor, [sub1.eventBuffer count]);
NSLog(@"Sub2 cursor: %d, events: %d", sub2.cursor, [sub2.eventBuffer count]);

// Unsubscribe sub2
[fh unsubscribe:@"appview-1"];

// Broadcast more
[fh broadcastEventWithType:@"identity" did:@"did:plc:abc" data:@{@"handle": @"alice.bsky.social"}];

NSLog(@"Sub1 events: %d", [sub1.eventBuffer count]);
NSLog(@"Sub2 events (unchanged): %d", [sub2.eventBuffer count]);

// Late subscriber joins and replays from cursor 0
FirehoseSubscriber *late = [[FirehoseSubscriber alloc] initWithId:@"relay-2"];
[fh subscribe:late];
[fh replayFromCursor:0 toSubscriber:late];
NSLog(@"Late subscriber events: %d", [late.eventBuffer count]);

## Step 6: Repository Diff

The firehose broadcasts commit events that contain MST diffs.
A diff compares two MST snapshots and produces operations:
add (new key), update (changed CID), or delete (removed key).

In [ ]:
@interface MSTSnapshot : NSObject
@property (nonatomic, strong) NSMutableDictionary *entries;
- (void)put:(NSString *)key cid:(NSString *)cid;
- (NSString *)get:(NSString *)key;
- (void)remove:(NSString *)key;
- (NSMutableArray *)allEntries;
- (NSString *)description;
@end

@implementation MSTSnapshot
- (instancetype)init {
    self = [super init];
    if (self) { _entries = [NSMutableDictionary dictionary]; }
    return self;
}
- (void)put:(NSString *)key cid:(NSString *)cid {
    [_entries setObject:cid forKey:key];
}
- (NSString *)get:(NSString *)key {
    return [_entries objectForKey:key];
}
- (void)remove:(NSString *)key {
    [_entries removeObjectForKey:key];
}
- (NSMutableArray *)allEntries {
    NSMutableArray *result = [NSMutableArray array];
    for (NSString *key in _entries) {
        [result addObject:@{@"key": key, @"cid": [_entries objectForKey:key]}];
    }
    return result;
}
- (NSString *)description {
    return [NSString stringWithFormat:@"MSTSnapshot(%d entries)", [_entries count]];
}
@end

MSTSnapshot *snap = [[MSTSnapshot alloc] init];
[snap put:@"app.bsky.feed.post/abc" cid:@"bafyrei1111"];
[snap put:@"app.bsky.feed.post/def" cid:@"bafyrei2222"];
[snap put:@"app.bsky.actor.profile/self" cid:@"bafyrei3333"];
NSLog(@"%@", [snap description]);
NSLog(@"Post abc: %@", [snap get:@"app.bsky.feed.post/abc"]);

In [ ]:
@interface RepoDiff : NSObject
- (NSMutableArray *)diffFrom:(MSTSnapshot *)oldSnap to:(MSTSnapshot *)newSnap;
@end

@implementation RepoDiff
- (NSMutableArray *)diffFrom:(MSTSnapshot *)oldSnap to:(MSTSnapshot *)newSnap {
    NSMutableArray *ops = [NSMutableArray array];
    
    // Find adds and updates: keys in newSnap
    for (NSString *key in newSnap.entries) {
        NSString *newCid = [newSnap get:key];
        NSString *oldCid = [oldSnap get:key];
        if (oldCid == nil) {
            // New key → add
            [ops addObject:@{
                @"action": @"add",
                @"key": key,
                @"cid": newCid
            }];
        } else if (![newCid isEqualToString:oldCid]) {
            // Changed CID → update
            [ops addObject:@{
                @"action": @"update",
                @"key": key,
                @"prevCid": oldCid,
                @"cid": newCid
            }];
        }
    }
    
    // Find deletes: keys in oldSnap but not in newSnap
    for (NSString *key in oldSnap.entries) {
        if ([newSnap get:key] == nil) {
            [ops addObject:@{
                @"action": @"delete",
                @"key": key,
                @"prevCid": [oldSnap get:key]
            }];
        }
    }
    
    return ops;
}
@end

In [ ]:
@interface RepoSync : NSObject
- (MSTSnapshot *)applyDiff:(NSMutableArray *)diffOps toSnapshot:(MSTSnapshot *)snapshot;
@end

@implementation RepoSync
- (MSTSnapshot *)applyDiff:(NSMutableArray *)diffOps toSnapshot:(MSTSnapshot *)snapshot {
    MSTSnapshot *newSnap = [[MSTSnapshot alloc] init];
    // Copy existing entries
    for (NSString *key in snapshot.entries) {
        [newSnap put:key cid:[snapshot get:key]];
    }
    // Apply diff operations
    for (int i = 0; i < [diffOps count]; i++) {
        NSDictionary *op = [diffOps objectAtIndex:i];
        NSString *action = [op objectForKey:@"action"];
        NSString *key = [op objectForKey:@"key"];
        if ([action isEqualToString:@"add"] || [action isEqualToString:@"update"]) {
            [newSnap put:key cid:[op objectForKey:@"cid"]];
        } else if ([action isEqualToString:@"delete"]) {
            [newSnap remove:key];
        }
    }
    return newSnap;
}
@end

In [ ]:
// Old snapshot
MSTSnapshot *old = [[MSTSnapshot alloc] init];
[old put:@"app.bsky.feed.post/abc" cid:@"bafyrei1111"];
[old put:@"app.bsky.feed.post/def" cid:@"bafyrei2222"];
[old put:@"app.bsky.actor.profile/self" cid:@"bafyrei3333"];

// New snapshot (1 add, 1 update, 1 delete)
MSTSnapshot *new_ = [[MSTSnapshot alloc] init];
[new_ put:@"app.bsky.feed.post/abc" cid:@"bafyrei1111"];  // unchanged
[new_ put:@"app.bsky.feed.post/def" cid:@"bafyrei9999"];  // updated CID
[new_ put:@"app.bsky.feed.post/xyz" cid:@"bafyrei4444"];  // new key
// profile/self deleted

// Compute diff
RepoDiff *differ = [[RepoDiff alloc] init];
NSMutableArray *ops = [differ diffFrom:old to:new_];
NSLog(@"Diff operations: %d", [ops count]);
for (int i = 0; i < [ops count]; i++) {
    NSDictionary *op = [ops objectAtIndex:i];
    NSLog(@"  %@ %@", [op objectForKey:@"action"], [op objectForKey:@"key"]);
}

// Apply diff to old snapshot
RepoSync *syncer = [[RepoSync alloc] init];
MSTSnapshot *synced = [syncer applyDiff:ops toSnapshot:old];
NSLog(@"Synced: %@", [synced description]);

// Verify synced matches new
NSLog(@"Post abc: %@ (expect bafyrei1111)", [synced get:@"app.bsky.feed.post/abc"]);
NSLog(@"Post def: %@ (expect bafyrei9999)", [synced get:@"app.bsky.feed.post/def"]);
NSLog(@"Post xyz: %@ (expect bafyrei4444)", [synced get:@"app.bsky.feed.post/xyz"]);
NSLog(@"Profile: %@ (expect nil)", [synced get:@"app.bsky.actor.profile/self"]);

## Exercise

Add `reverseDiff:` to `RepoDiff` that inverts a diff — adds become deletes,
deletes become adds, and updates swap `prevCid` and `cid`. This is useful for
rolling back changes.

Hint: loop over the operations and create inverse operations for each.

In [ ]:
// Exercise: implement reverseDiff:
// Your code here...

## Solution

In [ ]:
// Solution: reverseDiff inverts each operation
@interface RepoDiff2 : NSObject
- (NSMutableArray *)reverseDiff:(NSMutableArray *)diffOps;
@end

@implementation RepoDiff2
- (NSMutableArray *)reverseDiff:(NSMutableArray *)diffOps {
    NSMutableArray *reversed = [NSMutableArray array];
    for (int i = 0; i < [diffOps count]; i++) {
        NSDictionary *op = [diffOps objectAtIndex:i];
        NSString *action = [op objectForKey:@"action"];
        NSString *key = [op objectForKey:@"key"];
        if ([action isEqualToString:@"add"]) {
            // Add becomes delete
            [reversed addObject:@{
                @"action": @"delete",
                @"key": key,
                @"prevCid": [op objectForKey:@"cid"]
            }];
        } else if ([action isEqualToString:@"delete"]) {
            // Delete becomes add
            [reversed addObject:@{
                @"action": @"add",
                @"key": key,
                @"cid": [op objectForKey:@"prevCid"]
            }];
        } else if ([action isEqualToString:@"update"]) {
            // Update swaps CIDs
            [reversed addObject:@{
                @"action": @"update",
                @"key": key,
                @"prevCid": [op objectForKey:@"cid"],
                @"cid": [op objectForKey:@"prevCid"]
            }];
        }
    }
    return reversed;
}
@end

// Test it: reverse the diff and apply it to new_ to get back to old
RepoDiff2 *rd2 = [[RepoDiff2 alloc] init];
NSMutableArray *revOps = [rd2 reverseDiff:ops];
NSLog(@"Reversed operations: %d", [revOps count]);
for (int i = 0; i < [revOps count]; i++) {
    NSDictionary *op = [revOps objectAtIndex:i];
    NSLog(@"  %@ %@", [op objectForKey:@"action"], [op objectForKey:@"key"]);
}

// Apply reverse diff to new_ should produce old
MSTSnapshot *rolledBack = [syncer applyDiff:revOps toSnapshot:new_];
NSLog(@"Post def after rollback: %@ (expect bafyrei2222)", [rolledBack get:@"app.bsky.feed.post/def"]);
NSLog(@"Profile after rollback: %@ (expect bafyrei3333)", [rolledBack get:@"app.bsky.actor.profile/self"]);

## What to Read Next

You now understand how ATProto synchronizes state across the network. Next:
- **Part 17: Migrations** — how database schemas evolve safely
- **Part 18: Archaeology — Repository** — how production MST/CAR/CBOR code works